# Step 16.1 — Acquire and Inspect Tiny Shakespeare：第一次把真实语料接到 MiniGPT 前面

Step 15 已经得到完整 MiniGPT：

$$
(B,T)\rightarrow(B,T,V).
$$

但到目前为止，输入 token IDs 主要还是 toy tensors。

Step 16 开始把模型接到真实文本。

本节只做三件事：

1. **Acquire / Cache**：准备 Tiny Shakespeare 下载函数与本地缓存路径；
2. **Encoding**：复用 Step 6 的 character tokenizer，把真实文本变成 integer token stream；
3. **Corpus Size**：检查 characters、UTF-8 bytes、vocabulary size 与 encoded token count。

本节**不做 train/validation split、不采样 batch、不训练模型**。

所有下载与检查代码都只写入母版，当前 Notebook 保持未执行。


## 1. 为什么现在选 Tiny Shakespeare？

我们需要一个比 toy string 更真实、但又足够小的 language-model corpus。

Tiny Shakespeare 是一个经典 character-level teaching corpus：

- 是一份连续文本文件；
- 有自然语言、换行、标点、speaker labels；
- 规模约 1.1M characters，个人电脑很容易处理；
- 适合我们当前已经学过的 character tokenizer。

本课程使用 Karpathy `char-rnn` 仓库中的：

```text
data/tinyshakespeare/input.txt
```

标准来源 URL：

```text
https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
```

Karpathy 的 nanoGPT 字符级准备脚本对标准文件报告：

```text
characters : 1,115,394
vocab size : 65
```

这些只作为 **reference expectation**。

真正练习时应以你本地缓存文件实际读取后的统计结果为准，而不是把参考数字硬写成 assertion。


## 2. Acquire 和 Cache 是两个不同动作

如果每次打开 Notebook 都重新联网下载语料，会带来：

- 不必要的网络等待；
- 网络失败导致课程不能继续；
- 同一数据重复下载。

因此使用：

```text
remote source
    ↓ first time only
local cache
    ↓ later runs
reuse local file
```

本项目缓存路径：

```text
data/tinyshakespeare/input.txt
```

仓库 `.gitignore` 已经忽略：

```text
/data/
```

所以这个本地语料不会被正常 Git 操作误提交到课程仓库。


## 3. 先定义 Acquisition Helper，再单独决定是否执行下载

为了避免“定义函数”和“真的联网下载”混在一起，本节把它们分成两个 cell。

### Definition-only cell

只定义：

```text
SOURCE_URL
CACHE_PATH
acquire_tiny_shakespeare(...)
```

运行 definition cell 本身不会下载。

### User-run acquisition cell

只有真正执行：

```python
corpus_path = acquire_tiny_shakespeare()
```

时才会：

- cache 已存在 → 直接复用；
- cache 不存在 → 联网下载一次并写入本地。

这和前面 15.5 区分 definition / usage 的原则一致。


In [ ]:
from pathlib import Path
from urllib.request import urlopen


SOURCE_URL = (
    'https://raw.githubusercontent.com/'
    'karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
)

DATA_DIR = Path('data/tinyshakespeare')
CACHE_PATH = DATA_DIR / 'input.txt'


def acquire_tiny_shakespeare(
    cache_path: Path = CACHE_PATH,
    source_url: str = SOURCE_URL,
) -> Path:
    """Return a local Tiny Shakespeare path, downloading only on cache miss."""

    cache_path = Path(cache_path)

    if cache_path.exists():
        print(f'cache hit: {cache_path}')
        return cache_path

    print(f'cache miss: {cache_path}')
    cache_path.parent.mkdir(parents=True, exist_ok=True)

    with urlopen(source_url, timeout=30) as response:
        raw_bytes = response.read()

    # Write to a temporary file first, then replace the final cache path.
    temp_path = cache_path.with_suffix('.tmp')
    temp_path.write_bytes(raw_bytes)
    temp_path.replace(cache_path)

    print(f'cached corpus: {cache_path}')
    return cache_path


# Definition only: this cell does NOT call the function.
# The master notebook remains unexecuted.


In [ ]:
# USER-RUN ACQUISITION CELL — prepared only, NOT executed in the master.

corpus_path = acquire_tiny_shakespeare()
print('corpus path:', corpus_path)

# First real execution may access the network if the cache is absent.
# Later executions should reuse data/tinyshakespeare/input.txt.


## 4. Cache 里保存的是 Bytes；语言模型先把它读成 Text

下载阶段得到的是 file bytes。

本节明确使用 UTF-8：

```python
text = corpus_path.read_text(encoding='utf-8')
```

读取后：

```text
type(text) == str
```

这时 `len(text)` 统计的是 Python string 中的 Unicode characters / code points 数量；

`len(text.encode('utf-8'))` 则统计重新编码后的 UTF-8 bytes 数量。

对于这个英文为主的语料，两者规模接近，但：

> **character count 与 byte count 是两个不同概念。**

未来如果文本包含大量中文等多字节 UTF-8 字符，两者差异会明显。


In [ ]:
# USER-RUN INSPECTION CELL — requires corpus_path from the acquisition cell.

text = corpus_path.read_text(encoding='utf-8')

preview_chars = 500

print('python type:', type(text))
print('preview length requested:', preview_chars)
print('preview with escapes visible:')
print(repr(text[:preview_chars]))

# repr(...) makes '\n' and other whitespace boundaries easier to inspect.
# No corpus text has been read or printed in the master notebook.


## 5. “Corpus Size” 至少要检查哪些量？

对 character-level language model，本节关注四个基本统计量。

### 1. Number of characters

$$
N_{char}=len(text).
$$

### 2. UTF-8 byte size

$$
N_{byte}=len(text.encode('utf-8')).
$$

### 3. Character vocabulary size

$$
V=|\operatorname{unique}(text)|.
$$

### 4. Encoded token count

Character tokenizer 中：

$$
\boxed{N_{token}=N_{char}}
$$

因为每一个 character 恰好变成一个 token ID。

这里还没有 train / validation 的概念，所以检查的是**整份 corpus**。


In [ ]:
# USER-RUN CORPUS-SIZE CELL — requires text from the previous cell.

num_characters = len(text)
num_utf8_bytes = len(text.encode('utf-8'))
characters = sorted(set(text))
vocab_size = len(characters)
num_newlines = text.count('\n')

print(f'characters: {num_characters:,}')
print(f'UTF-8 bytes: {num_utf8_bytes:,}')
print(f'unique characters / vocab size: {vocab_size:,}')
print(f'newline characters: {num_newlines:,}')
print('vocabulary with escapes visible:')
print([repr(ch) for ch in characters])

# Reference for the standard Karpathy file:
# approximately/exactly as reported by the official nanoGPT preparation script:
# characters = 1,115,394
# vocab size = 65
# Treat the locally inspected file as the operational source of truth.
# This cell has NOT been executed in the master notebook.


## 6. 把 Step 6 的 Character Tokenizer 用到真实 Corpus

Step 6.1 已经学过：

```text
unique characters
↓
sorted vocabulary
↓
stoi: character -> integer ID
itos: integer ID -> character
↓
encode / decode
```

现在只是把 toy string 换成整份 Tiny Shakespeare。

定义：

$$
\mathcal V
=
sorted(unique(text)).
$$

如果：

$$
V=|\mathcal V|,
$$

那么 token IDs 的合法范围是：

$$
\boxed{0,1,\ldots,V-1}.
$$

换行 `\n`、空格、标点、大写/小写字符都可以成为各自独立 token。

本课程这个 minimal character tokenizer 没有 `<UNK>`；因为 vocabulary 直接从当前 corpus 建立，所以完整 corpus 自身一定都可编码。

如果以后拿一个包含 unseen character 的外部字符串调用 `encode`，当前实现会用 `KeyError` 明确暴露 vocabulary mismatch。


In [ ]:
# USER-RUN TOKENIZER-DEFINITION CELL — requires text.

characters = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(characters)}
itos = {i: ch for i, ch in enumerate(characters)}


def encode(s: str) -> list[int]:
    return [stoi[ch] for ch in s]


def decode(token_ids: list[int]) -> str:
    return ''.join(itos[token_id] for token_id in token_ids)


print('vocab size:', len(characters))
print('first vocabulary entries:', [(i, repr(ch)) for i, ch in enumerate(characters[:12])])

# Definitions / inspection only. No encoding of the full corpus occurs yet.


## 7. 编码前先做一个小 Round-Trip Sanity Check

Tokenizer 最基本的 contract 是：

$$
\boxed{
decode(encode(s))=s
}
$$

只要字符串 `s` 的每个 character 都在当前 vocabulary 中。

例如从 corpus 自身取一个短 preview：

```text
sample text
↓ encode
[integer IDs]
↓ decode
same sample text
```

这里测试的是 tokenizer mapping 是否可逆，不是在测试 language model。


In [ ]:
# PREPARED ROUND-TRIP CHECK — NOT executed in the master.

sample_text = text[:120]
sample_ids = encode(sample_text)
reconstructed = decode(sample_ids)

assert reconstructed == sample_text

print('sample characters:', len(sample_text))
print('sample token count:', len(sample_ids))
print('first token IDs:', sample_ids[:40])
print('round-trip reconstructed preview:', repr(reconstructed))

# This assertion has NOT been run in the master notebook.


## 8. 编码整份 Corpus：从 `str` 变成一条 1-D Token Stream

完整语料：

```text
text: Python str
```

Character encoding 后：

$$
ids=[id_0,id_1,\ldots,id_{N-1}].
$$

因为一个 character 对应一个 token：

$$
N=len(text).
$$

为了后续 MLX data pipeline，可以再转换为：

$$
token\_stream\in\mathbb Z^N.
$$

本节使用：

```python
mx.array(encoded_ids, dtype=mx.int32)
```

所以 shape 为：

$$
\boxed{(N,)}.
$$

注意现在还只是**一条连续的 token stream**。

本节不会把它切成 train / validation，也不会产生 `(B,T)` batches。


In [ ]:
import mlx.core as mx


# USER-RUN FULL-ENCODING CELL — prepared only, NOT executed in the master.

encoded_ids = encode(text)  # Python list[int], length N
token_stream = mx.array(encoded_ids, dtype=mx.int32)  # (N,)

assert len(encoded_ids) == len(text)
assert token_stream.shape == (len(text),)

print('encoded token count:', len(encoded_ids))
print('token stream shape:', token_stream.shape)
print('token stream dtype:', token_stream.dtype)
print('min valid ID should be >= 0')
print('max valid ID should be < vocab_size =', len(characters))

# No full-corpus encoding or assertion has run in the master notebook.


## 9. 从 Corpus Scale 到未来 Model Context：N 和 T 不是一回事

现在有两个很容易混淆的长度。

### Corpus length

$$
N\approx1.1\text{ million character tokens}.
$$

它描述整份语料有多长。

### Model context length

$$
T\le T_{max}.
$$

它描述一次 MiniGPT forward 能看到多长的 sequence window。

一般：

$$
\boxed{N\gg T}.
$$

所以训练不会把整个 1.1M-token corpus 一次性作为 `(1,N)` 输入 MiniGPT。

后面需要从长 token stream 中取有限长度 windows，再组成 batches。

但：

> **如何 split、怎样定义 block_size、怎样 sampler batch，全部属于 16.2/16.3，当前不进入。**


## 10. Reference Expectation 与 Local Source of Truth

标准 Karpathy / nanoGPT Tiny Shakespeare 字符版通常报告：

```text
characters = 1,115,394
vocab size = 65
```

这两个数非常适合作为 smoke-check reference。

但课程代码不写：

```python
assert len(text) == 1_115_394
assert vocab_size == 65
```

原因是 acquisition source 未来如果内容或换行编码发生变化，课程不应因为一个参考数字而错误失败。

因此规则是：

```text
reference numbers
    -> help you notice an unexpected file

actual local read/inspection
    -> operational source of truth
```


## 11. 在 Work Copy 中的实际操作顺序

当你之后真正练习 16.1 时，从上到下：

```text
1. run acquisition-helper definition
2. run acquisition usage cell
   -> cache hit or one-time network download
3. read UTF-8 text + inspect preview
4. inspect corpus size / vocabulary
5. define stoi / itos / encode / decode
6. run short round-trip check
7. encode full corpus into 1-D MLX token stream
```

运行完成后你应该拥有：

```text
text          : str
characters    : list[str]
stoi / itos   : mappings
encoded_ids   : list[int]
token_stream  : MLX integer array, shape (N,)
```

但此时仍然**没有**：

```text
train_data
val_data
batch sampler
x/y shifted batches
training loop
```

这些留给后续既有小节。


## 12. Step 16.1 材料边界

- Tiny Shakespeare 使用 Karpathy `char-rnn/data/tinyshakespeare/input.txt` 作为远端来源，缓存到本地 `data/tinyshakespeare/input.txt`。
- 仓库 `.gitignore` 已忽略 `/data/`，因此本地 corpus cache 不应正常进入 Git commit。
- acquisition helper 采用 cache-first：存在本地文件就不联网；cache miss 才通过 Python standard-library `urllib.request.urlopen` 获取。
- 下载 helper definition 与真正 acquisition usage 分离；母版本轮未调用下载函数。
- 文本明确以 UTF-8 读取，并区分 character count 与 byte count。
- 复用 Step 6 character tokenizer：`sorted(set(text)) -> stoi/itos -> encode/decode`；newline、space、punctuation 都是普通 character tokens。
- character-level encoding 中 `N_token = N_char`；完整 encoded corpus 是 shape `(N,)` 的一条 1-D integer token stream。
- 标准文件的 `1,115,394 characters / 65 vocabulary` 只作为 reference expectation，不作为硬断言；local inspection 是实际 authority。
- 本轮只准备 Step 16.1，修正计数从 0/2 起；不进入 Step 16.2，不切 train/validation、不创建 batch sampler、不下载数据、不执行 Notebook/检查/训练。
